# Ingest plan — programmatic fetch (StatCan WDS + CMHC)

This notebook searches StatCan's Web Data Service (WDS) for relevant product IDs (PIDs) and scrapes CMHC table pages for CSV links, then downloads matching CSVs to `data/raw/`. Run cells in order.

In [13]:
# Cell 1: setup and imports
import requests
import re
import json
import time
from pathlib import Path
import pandas as pd

PROJECT = Path('.')
RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# StatCan WDS base endpoints (see https://www150.statcan.gc.ca/t1/wds)
STATCAN_WDS_BASE = 'https://www150.statcan.gc.ca/t1/wds'
ALL_CUBES_LITE = f'{STATCAN_WDS_BASE}/en/grp/wds/fn/getAllCubesListLite'
FULL_TABLE_CSV = f'{STATCAN_WDS_BASE}/en/grp/wds/fn/getFullTableDownloadCSV'  # append /{PID}/en

# Simple retrying HTTP helper to handle transient 404/403/timeouts
SESSION = requests.Session()
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "*/*",
}

def request_with_retry(url, *, method="GET", retries=3, backoff=2, timeout=30, **kwargs):
    headers = kwargs.pop("headers", {})
    merged_headers = {**DEFAULT_HEADERS, **headers}
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.request(method=method, url=url, headers=merged_headers, timeout=timeout, **kwargs)
            if resp.status_code in (403, 404) and attempt < retries:
                time.sleep(backoff * attempt)
                continue
            resp.raise_for_status()
            return resp
        except Exception as exc:
            last_exc = exc
            if attempt == retries:
                break
            time.sleep(backoff * attempt)
    raise last_exc # type: ignore

print('setup complete')

setup complete


In [14]:
# Quick starter: fetch a public CSV (no WDS required)
sample_csv_url = "https://raw.githubusercontent.com/plotly/datasets/master/2014_usa_states.csv"
sample_dest = RAW / "starter_sample.csv"

if not sample_dest.exists():
    r = request_with_retry(sample_csv_url, timeout=60, stream=True)
    with open(sample_dest, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
    print("downloaded", sample_dest)
else:
    print(sample_dest, "exists")

# Load and preview
sample_df = pd.read_csv(sample_dest)
sample_df.head()

downloaded data/raw/starter_sample.csv


,Rank,State,Postal,Population
0,1,Alabama,AL,4849377.0
1,2,Alaska,AK,736732.0
2,3,Arizona,AZ,6731484.0
3,4,Arkansas,AR,2966369.0
4,5,California,CA,38802500.0


# Note on WDS availability
As of now the StatCan WDS REST/SOAP endpoints listed on the docs are returning 404s after redirects. Until they come back, use direct table CSV downloads (from the table viewer’s Download → CSV) or any other CSV source to test the pipeline. The demo cell below fetches a public CSV so you can verify the flow without relying on WDS.

In [15]:
# Cell 2: helpers for StatCan WDS
_statcan_cubes_cache = None

def fetch_all_statcan_cubes(force=False):
    global _statcan_cubes_cache
    if _statcan_cubes_cache is not None and not force:
        return _statcan_cubes_cache
    try:
        resp = request_with_retry(ALL_CUBES_LITE, method="POST", retries=4, backoff=2, timeout=45, json={})
    except Exception as first_exc:
        # Fallback to GET if POST is rejected (some StatCan edges return 404 on POST)
        resp = request_with_retry(ALL_CUBES_LITE, method="GET", retries=4, backoff=2, timeout=45)
    data = resp.json()
    # data expected as list of cube metadata objects with 'title' and 'pid'
    _statcan_cubes_cache = data
    return data

def statcan_search(keyword, max_results=10):
    keyword = keyword.lower()
    cubes = fetch_all_statcan_cubes()
    matches = []
    for c in cubes:
        title = (c.get('title') or c.get('productTitle') or '')
        pid = c.get('pid') or c.get('productId') or c.get('productID')
        if not title or not pid:
            continue
        if keyword in title.lower():
            matches.append({'pid': pid, 'title': title})
        if len(matches) >= max_results:
            break
    return matches

def download_statcan_table_csv(pid, dest_path):
    url = f'{FULL_TABLE_CSV}/{pid}/en'
    dest = Path(dest_path)
    if dest.exists():
        print(dest, 'exists')
        return dest
    print('downloading', url)
    try:
        resp = request_with_retry(url, method="POST", retries=4, backoff=2, timeout=90, stream=True, json={})
    except Exception as first_exc:
        resp = request_with_retry(url, method="GET", retries=4, backoff=2, timeout=90, stream=True)
    with open(dest, 'wb') as f:
        for chunk in resp.iter_content(1024*1024):
            if chunk:
                f.write(chunk)
    print('wrote', dest)
    return dest

In [16]:
# Cell 3: helpers for CMHC page scraping and generic downloading
def find_csv_links_on_page(page_url):
    try:
        r = request_with_retry(page_url, retries=4, backoff=2, timeout=45)
    except Exception as e:
        print('failed to fetch', page_url, e)
        return []
    html = r.text
    # find href="..." values
    pattern = r"href=[\"']([^\"']+)[\"']"
    hrefs = re.findall(pattern, html, flags=re.I)
    candidates = []
    for h in hrefs:
        if h.lower().endswith('.csv') or h.lower().endswith('.xlsx') or ('download' in h.lower() and ('.csv' in h.lower() or '.xlsx' in h.lower())):
            # make absolute if needed
            if h.startswith('//'):
                h = 'https:' + h
            elif h.startswith('/'):
                base = re.match(r'(https?://[^/]+)', page_url)
                if base:
                    h = base.group(1) + h
            candidates.append(h)
    # dedupe
    seen = []
    out = []
    for c in candidates:
        if c not in seen:
            seen.append(c)
            out.append(c)
    return out

def download_file(url, dest_path):
    dest = Path(dest_path)
    if dest.exists():
        print(dest, 'exists')
        return dest
    print('downloading', url)
    r = request_with_retry(url, retries=4, backoff=2, timeout=90, stream=True)
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(1024*1024):
            if chunk:
                f.write(chunk)
    print('wrote', dest)
    return dest

In [17]:
# Cell 4: dataset definitions (keywords / CMHC pages)
datasets = [
    { 'name': 'median_household_income', 'provider': 'statcan', 'keyword': 'median total income' },
    { 'name': 'population_estimates', 'provider': 'statcan', 'keyword': 'population estimates cma' },
    { 'name': 'unemployment_rate', 'provider': 'statcan', 'keyword': 'unemployment rate cma' },
    { 'name': 'cpi_all_items', 'provider': 'statcan', 'keyword': 'consumer price index all-items' },
    {
        'name': 'rental_market_rents',
        'provider': 'cmhc',
        'direct_url': 'https://assets.cmhc-schl.gc.ca/sites/cmhc/professional/housing-markets-data-and-research/housing-data-tables/rental-market/rental-market-report-data-tables/2025/rmr-canada-2025-en.xlsx?rev=11bbba5e-64a5-4dcd-a81f-7252c2b12537&_gl=1*529jeb*_gcl_au*NzkzNDcyMTcxLjE3NjY2MTc2NDU.*_ga*NTc3NDE3MTQyLjE3NjY2MTc2NDY.*_ga_CY7T7RT5C4*czE3NjY2MTc2NDYkbzEkZzEkdDE3NjY2MTc2NzQkajMyJGwwJGgw'
    },
    {
        'name': 'housing_starts',
        'provider': 'cmhc',
        'page': 'https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/housing-market-data'
    }
]

print('datasets defined')

datasets defined


In [ ]:
# Cell 5: run fetch for each dataset (statcan via WDS, cmhc via page scrape)
# TODO - revisit dataset import once WDS API stabilizes
for ds in datasets:
    name = ds["name"]
    try:
        if ds["provider"] == "statcan":
            print("\nSearching StatCan for", name, "keyword=", ds.get("keyword"))
            matches = statcan_search(ds.get("keyword", ""), max_results=10)

            if not matches:
                print("no matches for", name)
                continue

            # show top candidates
            for i, m in enumerate(matches[:5], 1):
                print(i, m["pid"], m["title"])

            pid = matches[0]["pid"]
            dest = RAW / f"{name}.csv"
            try:
                download_statcan_table_csv(pid, dest)
            except Exception as e:
                print("statcan download failed", e)

            time.sleep(1)

        elif ds["provider"] == "cmhc":
            direct_url = ds.get("direct_url")
            page = ds.get("page")

            if direct_url:
                print("Downloading CMHC direct file for", name)
                suffix_source = direct_url.split('?')[0]
                extension = suffix_source.split('.')[-1] if '.' in suffix_source else 'bin'
                dest = RAW / f"{name}.{extension}"
                try:
                    download_file(direct_url, dest)
                except Exception as e:
                    print("download failed", e)

            elif page:
                print("Searching CMHC page for", name, page)

                links = find_csv_links_on_page(page)
                if not links:
                    print("no csv links found on", page)
                    continue

                for link in links[:3]:
                    print("candidate:", link)

                dest = RAW / f"{name}.{links[0].split('.')[-1]}"
                try:
                    download_file(links[0], dest)
                except Exception as e:
                    print("download failed", e)

                time.sleep(1)

            else:
                print("no CMHC direct_url or page defined for", name)

            time.sleep(1)

        else:
            print("unknown provider for", name, ds.get("provider"))

    except Exception as e:
        print("error processing", name, e)



Searching StatCan for median_household_income keyword= median total income


error processing median_household_income 404 Client Error:  for url: https://www150.statcan.gc.ca/t1/wds/en/grp/wds/fn/getAllCubesListLite

Searching StatCan for population_estimates keyword= population estimates cma
error processing population_estimates 404 Client Error:  for url: https://www150.statcan.gc.ca/t1/wds/en/grp/wds/fn/getAllCubesListLite

Searching StatCan for unemployment_rate keyword= unemployment rate cma
error processing unemployment_rate 404 Client Error:  for url: https://www150.statcan.gc.ca/t1/wds/en/grp/wds/fn/getAllCubesListLite

Searching StatCan for cpi_all_items keyword= consumer price index all-items
error processing cpi_all_items 404 Client Error:  for url: https://www150.statcan.gc.ca/t1/wds/en/grp/wds/fn/getAllCubesListLite
data/raw/rental_market_rents.xlsx exists
Searching CMHC page for housing_starts https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/housing-market-data
no csv links found on https://www.c

In [19]:
# Cell 6: quick QA of downloaded raw files
import glob
for p in glob.glob(str(RAW / '*')):
    try:
        size = Path(p).stat().st_size
        print(p, 'size=', size)
    except Exception as e:
        print('stat failed', p, e)

data/raw/starter_sample.csv size= 1411
data/raw/rental_market_rents.xlsx size= 74401
